# Stage 2: Attack-Relation Brain (Redesigned for Local Ollama)

This version targets a small, self-hosted Ollama cluster (e.g. one GPU laptop + one Mac)
and is engineered to finish a 500-topic run in hours instead of days. The key changes
vs the original:

1. **Embedding-based pair prefilter.** Uses `sentence-transformers` on CPU to compute
   pairwise similarity per topic. Pairs that are obviously unrelated (large round gap,
   very low similarity, same-stance same-round with low similarity) are auto-labeled
   `None` without ever hitting the LLM. Typically cuts pair-classification calls by
   60-80%.

2. **Batched LLM calls.** Pair classifications run 6 per prompt; strength scoring runs
   10 per prompt. Single LLM call returns a JSON array. Amortizes per-call overhead
   by roughly 5-8x. On parse failure, the batch falls back to per-item calls so one
   malformed array doesn't nuke all its results.

3. **No rate-limit machinery firing.** Ollama endpoints have no `rpm` field, so the
   pacing code is a no-op. The circuit breaker is still there to handle crashed
   endpoints.


In [ ]:
import os, json, time, asyncio, logging, statistics, random
from pathlib import Path
from datetime import datetime
from itertools import permutations
from dataclasses import dataclass, field
from typing import Any, Optional

import httpx
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists()), cwd)
ENV_PATH = PROJECT_ROOT / '.env'
load_dotenv(ENV_PATH if ENV_PATH.exists() else None)

# -----------------------------------------------------------------------------
# Endpoint pool configuration (same format as before)
# -----------------------------------------------------------------------------
def _load_endpoints() -> list[dict]:
    raw_json = os.environ.get('MAJ_ENDPOINTS_JSON', '').strip()
    if raw_json:
        return json.loads(raw_json)

    for candidate in (PROJECT_ROOT / 'endpoints.json', cwd / 'endpoints.json'):
        if candidate.exists():
            with open(candidate, 'r', encoding='utf-8') as f:
                return json.load(f)

    # Fallback: build from individual env vars.
    configs = []
    ollama_urls = [u.strip() for u in os.environ.get('MAJ_OLLAMA_URLS', '').split(',') if u.strip()]
    if ollama_urls:
        ollama_model = os.environ.get('MAJ_OLLAMA_MODEL', 'gemma3:12b')
        ollama_conc = int(os.environ.get('MAJ_OLLAMA_CONCURRENCY', '2'))
        ollama_timeout = float(os.environ.get('MAJ_OLLAMA_TIMEOUT', '180'))
        for i, url in enumerate(ollama_urls, start=1):
            configs.append({
                'name': f'ollama_{i}',
                'type': 'ollama',
                'base_url': url.rstrip('/'),
                'model': ollama_model,
                'concurrency': ollama_conc,
                'timeout': ollama_timeout,
            })
    if not configs:
        raise ValueError(
            'No endpoints configured. Create endpoints.json or set MAJ_OLLAMA_URLS.'
        )
    return configs


ENDPOINT_CONFIGS = _load_endpoints()

# Resolve "env:VAR_NAME" references in api_key fields.
for _ec in ENDPOINT_CONFIGS:
    key = _ec.get('api_key')
    if isinstance(key, str) and key.startswith('env:'):
        resolved = os.environ.get(key[4:])
        if not resolved:
            print(f"WARNING: endpoint {_ec['name']!r} references env var {key[4:]} "
                  f"which is unset; endpoint will run unauthenticated")
            _ec['api_key'] = None
        else:
            _ec['api_key'] = resolved

# -----------------------------------------------------------------------------
# Run-level config
# -----------------------------------------------------------------------------
TEMPERATURE = float(os.environ.get('MAJ_STAGE2_TEMPERATURE', '0.2'))
MAX_TOKENS = int(os.environ.get('MAJ_STAGE2_MAX_TOKENS', '1200'))  # bumped for batch responses
CONFIDENCE_THRESHOLD = float(os.environ.get('MAJ_STAGE2_CONFIDENCE', '0.65'))
ATTACK_MODE = os.environ.get('MAJ_STAGE2_ATTACK_MODE', 'targeted')
TOPIC_LIMIT = int(os.environ.get('MAJ_STAGE2_TOPIC_LIMIT', '0'))
EVAL_SPLIT = os.environ.get('MAJ_EVAL_SPLIT', 'ddo_sample')

MAX_TASK_ATTEMPTS = int(os.environ.get('MAJ_STAGE2_TASK_ATTEMPTS', '4'))

CB_FAIL_THRESHOLD = int(os.environ.get('MAJ_STAGE2_CB_FAILS', '3'))
CB_COOLDOWN_BASE = float(os.environ.get('MAJ_STAGE2_CB_COOLDOWN', '15'))
CB_COOLDOWN_MAX = float(os.environ.get('MAJ_STAGE2_CB_COOLDOWN_MAX', '300'))
TOPIC_CONCURRENCY = int(os.environ.get('MAJ_STAGE2_TOPIC_CONCURRENCY', '16'))

# -----------------------------------------------------------------------------
# NEW: Prefilter + batching config
# -----------------------------------------------------------------------------
# Embedding model used to compute per-topic pair similarity. all-MiniLM-L6-v2 is
# ~80 MB, runs on CPU, and encodes thousands of short texts per second. Good
# enough to tell "these two arguments are about totally different subpoints"
# from "these two talk about the same thing".
EMBED_MODEL_NAME = os.environ.get('MAJ_STAGE2_EMBED_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')

# Prefilter thresholds. Tune by spot-checking which pairs get auto-None'd on a
# handful of topics before trusting them for the whole run.
PREFILTER_ENABLED = os.environ.get('MAJ_STAGE2_PREFILTER', '1') not in {'0', 'false', 'False'}
PF_MAX_ROUND_GAP = int(os.environ.get('MAJ_STAGE2_PF_MAX_ROUND_GAP', '4'))
PF_MIN_SIMILARITY = float(os.environ.get('MAJ_STAGE2_PF_MIN_SIM', '0.15'))
PF_SAME_STANCE_MIN_SIM = float(os.environ.get('MAJ_STAGE2_PF_SAME_STANCE_MIN_SIM', '0.35'))

# Batching config. PAIR_BATCH_SIZE=6 keeps the prompt under a few thousand
# tokens for Gemma 3 12B even with long arguments; raise if you see the model
# coping fine with more, lower if you see truncated JSON.
PAIR_BATCH_SIZE = int(os.environ.get('MAJ_STAGE2_PAIR_BATCH', '6'))
STRENGTH_BATCH_SIZE = int(os.environ.get('MAJ_STAGE2_STRENGTH_BATCH', '10'))

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://YOUR_VM_IP:5000')

IN_FILE = PROJECT_ROOT / 'outputs' / 'stage1' / EVAL_SPLIT / 'stage1_arguments.json'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'stage2' / EVAL_SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'stage2_relations.json'
FAILURES_FILE = OUT_DIR / 'stage2_failures.json'
RESUME_EXISTING = os.environ.get('MAJ_STAGE2_RESUME', '1').strip() not in {'0', 'false', 'False'}
CONTINUE_ON_ERROR = os.environ.get('MAJ_STAGE2_CONTINUE_ON_ERROR', '1').strip() not in {'0', 'false', 'False'}

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('stage2')

print(f'.env loaded from     : {ENV_PATH if ENV_PATH.exists() else "not found"}')
print(f'Eval split           : {EVAL_SPLIT}')
print(f'Attack mode          : {ATTACK_MODE}')
print(f'Confidence threshold : {CONFIDENCE_THRESHOLD}')
print(f'Input                : {IN_FILE.resolve()}')
print(f'Output               : {OUT_FILE.resolve()}')
print(f'Prefilter enabled    : {PREFILTER_ENABLED} '
      f'(round_gap<={PF_MAX_ROUND_GAP}, cos>={PF_MIN_SIMILARITY}, '
      f'same_stance_cos>={PF_SAME_STANCE_MIN_SIM})')
print(f'Pair batch size      : {PAIR_BATCH_SIZE}')
print(f'Strength batch size  : {STRENGTH_BATCH_SIZE}')
print()
print('Endpoint pool:')
for ec in ENDPOINT_CONFIGS:
    tag = f"[{ec['type']}]"
    auth = ' (authed)' if ec.get('api_key') else ''
    rpm = f", rpm={ec['rpm']}" if ec.get('rpm') else ''
    print(f"  - {ec['name']:20s} {tag:10s} {ec['base_url']:40s} model={ec['model']} conc={ec.get('concurrency', 2)}{rpm}{auth}")


## 1. Prompts (batched and single-item variants)

In [ ]:
LABELS = ['Attack', 'Support', 'Neutral', 'None']


# ---------- Single-item prompts (fallback path) ------------------------------

def build_relation_prompt(topic_text, arg_a, arg_b, attack_mode='targeted'):
    targeted = 'If labeling Attack, quote or paraphrase the attacked premise from argument B in `attacked_premise`.' if attack_mode == 'targeted' else ''
    return f"""You are the Stage 2 attack-relation brain for MAJ-Debate.

Topic: {topic_text}

Argument A ({arg_a['arg_id']}, {arg_a['stance']}, round {arg_a['round']}):
{arg_a['text']}

Argument B ({arg_b['arg_id']}, {arg_b['stance']}, round {arg_b['round']}):
{arg_b['text']}

Task:
1. Classify the ordered pair (A -> B) as exactly one of: Attack, Support, Neutral, None.
2. Give a one-sentence justification grounded in the two arguments.
3. Return a confidence value between 0 and 1.
4. Return an attacked_premise string when applicable, else null.

Rules:
- Prefer Attack only when A directly rebuts, undermines, or defeats B.
- Prefer Support only when A materially strengthens B.
- Use Neutral for weak relation and None for unrelated pairs.
- {targeted}
- Output only valid JSON.

Expected JSON:
{{"label": "Attack", "confidence": 0.83, "justification": "A disputes B\'s fairness premise.", "attacked_premise": "B assumes fairness follows from automation."}}"""


def build_strength_prompt(topic_text, arg_rec):
    return f"""You are calibrating argument strength for MAJ-Debate Stage 2 using a convincingness-style rubric.

Topic: {topic_text}
Argument ({arg_rec['arg_id']}, {arg_rec['stance']}, round {arg_rec['round']}):
{arg_rec['text']}

Score the argument from 0 to 1 using clarity, specificity, plausibility, and persuasive force.
Return JSON only:
{{"strength": 0.74, "rationale": "Clear claim with specific mechanism and moderate plausibility."}}"""


# ---------- Batched prompts (primary path) -----------------------------------

def build_batch_relation_prompt(topic_text, pairs, attack_mode='targeted'):
    """pairs is a list of (arg_a, arg_b) tuples. Returns a prompt that asks
    the model to classify all of them in one go and return a JSON array."""
    pair_blocks = []
    for idx, (a, b) in enumerate(pairs):
        pair_blocks.append(
            f"PAIR {idx}:\n"
            f"  A ({a['arg_id']}, {a['stance']}, r{a['round']}): {a['text']}\n"
            f"  B ({b['arg_id']}, {b['stance']}, r{b['round']}): {b['text']}"
        )
    pairs_text = '\n\n'.join(pair_blocks)
    targeted_note = 'For Attack, include attacked_premise (a short quote/paraphrase from B).' if attack_mode == 'targeted' else 'For any label, attacked_premise may be null.'

    return f"""You are the Stage 2 attack-relation brain for MAJ-Debate.

Topic: {topic_text}

For each ordered pair (A -> B) below, classify the relation as exactly one of:
- Attack  : A directly rebuts, undermines, or defeats B
- Support : A materially strengthens B
- Neutral : weak relation
- None    : unrelated

{targeted_note}

{pairs_text}

Return ONLY a JSON array, one object per pair, in the SAME ORDER. Do not include
any preamble or commentary. Schema per object:
{{"pair": <int index>, "label": "Attack|Support|Neutral|None", "confidence": <0.0-1.0>, "justification": "<one sentence>", "attacked_premise": "<string or null>"}}

Example for a 2-pair batch:
[
  {{"pair": 0, "label": "Attack", "confidence": 0.82, "justification": "A disputes B\'s fairness premise.", "attacked_premise": "B assumes fairness follows from automation."}},
  {{"pair": 1, "label": "None", "confidence": 0.55, "justification": "Arguments address unrelated subtopics.", "attacked_premise": null}}
]"""


def build_batch_strength_prompt(topic_text, args):
    """args is a list of argument records. Returns a prompt asking the model
    to score all of them in one go and return a JSON array."""
    arg_blocks = []
    for idx, a in enumerate(args):
        arg_blocks.append(
            f"ARG {idx} ({a['arg_id']}, {a['stance']}, r{a['round']}):\n{a['text']}"
        )
    args_text = '\n\n'.join(arg_blocks)

    return f"""You are calibrating argument strength for MAJ-Debate Stage 2.

Topic: {topic_text}

For EACH argument below, score it from 0.0 to 1.0 using clarity, specificity,
plausibility, and persuasive force.

{args_text}

Return ONLY a JSON array, one object per argument, in the SAME ORDER:
[
  {{"arg": <int index>, "strength": <0.0-1.0>, "rationale": "<one sentence>"}},
  ...
]"""


NO_THINKING_INSTRUCTION = (
    'Do not think step by step. Do not include chain-of-thought, analysis, or commentary. '
    'Return only the requested JSON and nothing else.'
)


# ---------- JSON parsers -----------------------------------------------------

def parse_json_object(text):
    """Tolerant parser: strips markdown fences, smart quotes, extracts first {...} block."""
    text = text.strip().replace('\u201c', '"').replace('\u201d', '"').replace('\u2019', "'")
    if '```' in text:
        parts = text.split('```')
        if len(parts) >= 3:
            inner = parts[1]
            if inner.lstrip().lower().startswith('json'):
                inner = inner.lstrip()[4:]
            text = inner.strip()
    start = text.find('{')
    end = text.rfind('}') + 1
    if start < 0 or end <= start:
        raise ValueError(f'No JSON object found in model output: {text[:200]!r}')
    return json.loads(text[start:end])


def parse_json_array(text):
    """Tolerant parser for top-level JSON arrays. Strips code fences, smart quotes,
    finds the outermost [...] block."""
    text = text.strip().replace('\u201c', '"').replace('\u201d', '"').replace('\u2019', "'")
    if '```' in text:
        parts = text.split('```')
        if len(parts) >= 3:
            inner = parts[1]
            if inner.lstrip().lower().startswith('json'):
                inner = inner.lstrip()[4:]
            text = inner.strip()
    start = text.find('[')
    end = text.rfind(']') + 1
    if start < 0 or end <= start:
        raise ValueError(f'No JSON array found in model output: {text[:200]!r}')
    parsed = json.loads(text[start:end])
    if not isinstance(parsed, list):
        raise ValueError(f'Expected JSON array, got {type(parsed).__name__}')
    return parsed


## 2. Async endpoint pool (circuit breaker, multi-endpoint retry)

In [ ]:
class PermanentEndpointError(Exception):
    """Raised when an endpoint has failed in a way no retry can fix."""
    pass


class RateLimitedError(Exception):
    """Raised on HTTP 429. Backpressure, not failure."""
    pass


@dataclass
class Endpoint:
    name: str
    type: str
    base_url: str
    model: str
    concurrency: int = 2
    weight: float = 1.0
    timeout: float = 180.0
    api_key: Optional[str] = None
    rpm: Optional[int] = None
    headers: dict = field(default_factory=dict)

    sem: asyncio.Semaphore = field(init=False)
    inflight: int = field(default=0, init=False)
    total_ok: int = field(default=0, init=False)
    total_fail: int = field(default=0, init=False)
    consecutive_failures: int = field(default=0, init=False)
    quarantine_until: float = field(default=0.0, init=False)
    quarantine_level: int = field(default=0, init=False)
    disabled: bool = field(default=False, init=False)
    disabled_reason: str = field(default='', init=False)
    rpm_window: list = field(default_factory=list, init=False)
    rpm_lock: asyncio.Lock = field(init=False)

    def __post_init__(self):
        self.sem = asyncio.Semaphore(self.concurrency)
        self.rpm_lock = asyncio.Lock()

    @property
    def free_slots(self) -> int:
        return self.concurrency - self.inflight

    @property
    def healthy(self) -> bool:
        return not self.disabled and time.monotonic() >= self.quarantine_until

    def disable(self, reason: str):
        if not self.disabled:
            log.error('endpoint %s DISABLED for run: %s', self.name, reason)
        self.disabled = True
        self.disabled_reason = reason

    async def _wait_rpm(self):
        """Min-interval pacer. No-op for Ollama endpoints (no rpm set)."""
        if not self.rpm:
            return
        min_interval = 60.0 / self.rpm
        async with self.rpm_lock:
            now = time.monotonic()
            last = self.rpm_window[-1] if self.rpm_window else 0.0
            wait = (last + min_interval) - now
            if wait > 0:
                await asyncio.sleep(wait)
            self.rpm_window.append(time.monotonic())
            if len(self.rpm_window) > self.rpm:
                self.rpm_window = self.rpm_window[-self.rpm:]

    def _mark_ok(self):
        self.total_ok += 1
        self.consecutive_failures = 0
        if self.quarantine_level > 0:
            log.info('endpoint %s recovered after quarantine', self.name)
        self.quarantine_level = 0
        self.quarantine_until = 0.0

    def _mark_fail(self, reason: str):
        self.total_fail += 1
        self.consecutive_failures += 1
        if self.consecutive_failures >= CB_FAIL_THRESHOLD:
            self.quarantine_level += 1
            cooldown = min(CB_COOLDOWN_BASE * (2 ** (self.quarantine_level - 1)), CB_COOLDOWN_MAX)
            self.quarantine_until = time.monotonic() + cooldown
            log.warning(
                'endpoint %s QUARANTINED for %.0fs (level %d) after %d consecutive failures (%s)',
                self.name, cooldown, self.quarantine_level, self.consecutive_failures, reason,
            )
            self.consecutive_failures = 0

    def _build_payload(self, prompt: str, max_tokens: int) -> tuple[str, dict, dict]:
        headers = {'Content-Type': 'application/json', **self.headers}
        if self.api_key:
            headers['Authorization'] = f'Bearer {self.api_key}'

        if self.type == 'ollama':
            url = f'{self.base_url}/api/chat'
            body = {
                'model': self.model,
                'messages': [
                    {'role': 'system', 'content': NO_THINKING_INSTRUCTION},
                    {'role': 'user', 'content': prompt},
                ],
                'stream': False,
                'options': {
                    'temperature': TEMPERATURE,
                    'num_predict': max_tokens,
                },
                'think': False,
            }
            return url, headers, body

        # OpenAI-compatible
        url = f'{self.base_url}/chat/completions'
        body = {
            'model': self.model,
            'messages': [
                {'role': 'system', 'content': NO_THINKING_INSTRUCTION},
                {'role': 'user', 'content': prompt},
            ],
            'temperature': TEMPERATURE,
            'max_tokens': max_tokens,
            'reasoning': {'exclude': True, 'enabled': False},
        }
        return url, headers, body

    def _extract_content(self, data: dict) -> str:
        if self.type == 'ollama':
            msg = data.get('message', {}) or {}
            content = msg.get('content', '')
            if not content and 'response' in data:
                content = data.get('response', '')
            return (content or '').strip()

        choices = data.get('choices') or []
        if not choices:
            return ''
        msg = choices[0].get('message', {}) or {}
        content = msg.get('content', '')
        if isinstance(content, list):
            content = ''.join(part.get('text', '') for part in content if isinstance(part, dict))
        return (content or '').strip()

    async def call(self, client: httpx.AsyncClient, prompt: str, max_tokens: int) -> str:
        await self._wait_rpm()
        url, headers, body = self._build_payload(prompt, max_tokens)
        resp = await client.post(url, headers=headers, json=body, timeout=self.timeout)

        if resp.status_code == 429:
            retry_after = resp.headers.get('Retry-After')
            try:
                wait_s = float(retry_after) if retry_after else 10.0
            except ValueError:
                wait_s = 10.0
            await asyncio.sleep(wait_s)
            raise RateLimitedError(f'{self.name} 429 rate-limited, waited {wait_s:.1f}s')

        if resp.status_code in (500, 502, 503, 504):
            raise RuntimeError(f'{self.name} HTTP {resp.status_code}: {resp.text[:200]}')

        if resp.status_code in (400, 401, 402, 403, 404):
            raise PermanentEndpointError(
                f'{self.name} HTTP {resp.status_code}: {resp.text[:400]} '
                f'(model={self.model!r})'
            )

        if resp.status_code != 200:
            raise RuntimeError(
                f'{self.name} HTTP {resp.status_code}: {resp.text[:400]} '
                f'(model={self.model!r})'
            )

        try:
            data = resp.json()
        except ValueError:
            raise RuntimeError(f'{self.name} returned non-JSON body: {resp.text[:300]}')

        if isinstance(data, dict) and data.get('error'):
            err = data['error']
            err_msg = err.get('message', str(err)) if isinstance(err, dict) else str(err)
            err_code = err.get('code') if isinstance(err, dict) else None
            if err_code in (400, 401, 402, 403, 404):
                raise PermanentEndpointError(f'{self.name} error payload (code={err_code}): {err_msg}')
            raise RuntimeError(f'{self.name} error payload: {err_msg}')

        content = self._extract_content(data)
        if not content:
            finish = None
            if self.type == 'ollama':
                finish = data.get('done_reason') or data.get('done')
            else:
                finish = (data.get('choices') or [{}])[0].get('finish_reason')
            raise RuntimeError(f'{self.name} returned empty content (finish={finish!r})')
        return content


class EndpointPool:
    def __init__(self, configs: list[dict]):
        self.endpoints: list[Endpoint] = []
        for c in configs:
            allowed = {'name', 'type', 'base_url', 'model', 'concurrency', 'weight',
                       'timeout', 'api_key', 'rpm', 'headers'}
            kwargs = {k: v for k, v in c.items() if k in allowed}
            self.endpoints.append(Endpoint(**kwargs))
        self.client = httpx.AsyncClient(
            limits=httpx.Limits(max_connections=200, max_keepalive_connections=50)
        )

    async def aclose(self):
        await self.client.aclose()

    def _pick(self) -> Optional[Endpoint]:
        now = time.monotonic()
        healthy = [e for e in self.endpoints if now >= e.quarantine_until and not e.disabled]
        if not healthy:
            return None
        healthy.sort(key=lambda e: (e.free_slots, e.weight, random.random()), reverse=True)
        return healthy[0]

    async def _wait_for_any_healthy(self):
        while True:
            live = [e for e in self.endpoints if not e.disabled]
            if not live:
                raise RuntimeError('All endpoints permanently disabled')
            next_free = min((e.quarantine_until for e in live), default=0.0)
            wait = max(0.1, next_free - time.monotonic())
            log.warning('All endpoints quarantined; waiting %.1fs', wait)
            await asyncio.sleep(min(wait, 10.0))
            if any(e.healthy for e in self.endpoints):
                return

    async def call(self, prompt: str, task_tag: str = '', max_tokens: Optional[int] = None) -> str:
        """Dispatch one prompt. max_tokens defaults to MAX_TOKENS; callers can
        raise it for batched prompts that produce longer JSON arrays."""
        if max_tokens is None:
            max_tokens = MAX_TOKENS
        last_error: Optional[BaseException] = None
        tried_names: list[str] = []

        attempt = 0
        while attempt < MAX_TASK_ATTEMPTS:
            attempt += 1
            ep = self._pick()
            if ep is None:
                await self._wait_for_any_healthy()
                ep = self._pick()
                if ep is None:
                    raise RuntimeError(f'No healthy endpoints available (task={task_tag})')

            tried_names.append(ep.name)
            async with ep.sem:
                ep.inflight += 1
                try:
                    content = await ep.call(self.client, prompt, max_tokens)
                    ep._mark_ok()
                    return content
                except asyncio.CancelledError:
                    raise
                except PermanentEndpointError as ex:
                    last_error = ex
                    ep.disable(str(ex)[:200])
                    attempt -= 1
                except RateLimitedError as ex:
                    last_error = ex
                    attempt -= 1
                    log.info('task %s hit 429 on %s; retrying without penalty',
                             task_tag or '?', ep.name)
                except Exception as ex:
                    last_error = ex
                    ep._mark_fail(str(ex)[:120])
                    log.warning(
                        'task %s attempt %d/%d on %s failed: %s',
                        task_tag or '?', attempt, MAX_TASK_ATTEMPTS, ep.name, ex,
                    )
                finally:
                    ep.inflight -= 1

            await asyncio.sleep(0.2 + 0.3 * random.random())

        raise RuntimeError(
            f'All {MAX_TASK_ATTEMPTS} attempts failed for task={task_tag} '
            f'(tried={tried_names}): {last_error}'
        )

    def stats(self) -> list[dict]:
        return [{
            'name': e.name,
            'model': e.model,
            'ok': e.total_ok,
            'fail': e.total_fail,
            'disabled': e.disabled,
            'disabled_reason': e.disabled_reason if e.disabled else '',
            'quarantined': (not e.disabled) and time.monotonic() < e.quarantine_until,
            'quarantine_level': e.quarantine_level,
        } for e in self.endpoints]


POOL = EndpointPool(ENDPOINT_CONFIGS)
print(f'Pool initialised with {len(POOL.endpoints)} endpoints')


## 3. Pair prefilter (cuts ~70% of LLM calls before they're made)

In [ ]:
# sentence-transformers is loaded lazily on first use so notebook import
# doesn't download 80MB of model weights if the user is just inspecting the
# code. If import fails, we fall back to a no-op prefilter that lets everything
# through.

_EMBEDDER = None
_EMBEDDER_FAILED = False


def get_embedder():
    """Lazy-load the sentence-transformer. Returns None if unavailable."""
    global _EMBEDDER, _EMBEDDER_FAILED
    if _EMBEDDER is not None or _EMBEDDER_FAILED:
        return _EMBEDDER
    try:
        from sentence_transformers import SentenceTransformer
        log.info('Loading embedding model: %s', EMBED_MODEL_NAME)
        _EMBEDDER = SentenceTransformer(EMBED_MODEL_NAME, device='cpu')
        log.info('Embedder ready')
    except Exception as ex:
        _EMBEDDER_FAILED = True
        log.warning('Could not load embedder (%s); prefilter will be disabled', ex)
        _EMBEDDER = None
    return _EMBEDDER


def compute_similarity_matrix(args):
    """Return NxN cosine similarity matrix, or None if embedder unavailable."""
    embedder = get_embedder()
    if embedder is None:
        return None
    import numpy as np
    texts = [a['text'] for a in args]
    embs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(embs) @ np.asarray(embs).T


def prefilter_pairs(args, sim_matrix):
    """Split all ordered pairs into (keep_for_llm, auto_none) based on cheap heuristics.

    auto_none items come back as dicts with source/target/reason so we can
    still emit relation records for them (label=None, confidence=0, kept=False)
    without spending an LLM call.
    """
    keep = []
    auto_none = []
    if not PREFILTER_ENABLED or sim_matrix is None:
        # No prefilter: return every ordered pair for LLM classification.
        for a, b in permutations(args, 2):
            keep.append((a, b))
        return keep, auto_none

    for i, a in enumerate(args):
        for j, b in enumerate(args):
            if i == j:
                continue
            round_gap = abs(a.get('round', 0) - b.get('round', 0))
            same_stance = a.get('stance') == b.get('stance')
            cos = float(sim_matrix[i, j])

            reason = None
            if round_gap >= PF_MAX_ROUND_GAP:
                reason = f'round_gap={round_gap}'
            elif cos < PF_MIN_SIMILARITY:
                reason = f'low_sim={cos:.3f}'
            elif same_stance and round_gap == 0 and cos < PF_SAME_STANCE_MIN_SIM:
                reason = f'same_stance_low_sim={cos:.3f}'

            if reason is None:
                keep.append((a, b))
            else:
                auto_none.append({
                    'source_arg_id': a['arg_id'],
                    'target_arg_id': b['arg_id'],
                    'source_stance': a.get('stance'),
                    'target_stance': b.get('stance'),
                    'source_round': a.get('round'),
                    'target_round': b.get('round'),
                    'label': 'None',
                    'confidence': 0.0,
                    'kept': False,
                    'justification': f'PREFILTERED: {reason}',
                    'attacked_premise': None,
                    'prefiltered': True,
                    'cos_sim': round(cos, 3),
                })
    return keep, auto_none


## 4. Batched LLM calls (with per-item fallback on parse failure)

In [ ]:
def _coerce_label(raw):
    label = str(raw).strip().title()
    return label if label in LABELS else 'None'


def _coerce_confidence(raw):
    try:
        v = float(raw)
    except (TypeError, ValueError):
        return 0.0
    return max(0.0, min(1.0, v))


def _relation_base(a, b):
    return {
        'source_arg_id': a['arg_id'],
        'target_arg_id': b['arg_id'],
        'source_stance': a.get('stance'),
        'target_stance': b.get('stance'),
        'source_round': a.get('round'),
        'target_round': b.get('round'),
    }


def _relation_from_obj(a, b, obj):
    base = _relation_base(a, b)
    label = _coerce_label(obj.get('label', 'None'))
    confidence = _coerce_confidence(obj.get('confidence', 0.0))
    return {
        **base,
        'label': label,
        'confidence': round(confidence, 3),
        'kept': confidence >= CONFIDENCE_THRESHOLD and label != 'None',
        'justification': obj.get('justification', 'No justification returned.'),
        'attacked_premise': obj.get('attacked_premise'),
    }


def _relation_failure(a, b, err):
    base = _relation_base(a, b)
    return {
        **base,
        'label': 'None',
        'confidence': 0.0,
        'kept': False,
        'justification': f'CLASSIFY_FAILED: {err}',
        'attacked_premise': None,
        'failed': True,
    }


async def _classify_single_pair(topic_text, arg_a, arg_b, attack_mode):
    """Fallback path: one pair, one LLM call. Used when a batch fails to parse."""
    tag = f"pair[{arg_a['arg_id']}->{arg_b['arg_id']}]"
    try:
        raw = await POOL.call(
            build_relation_prompt(topic_text, arg_a, arg_b, attack_mode=attack_mode),
            task_tag=tag,
            max_tokens=400,
        )
        return _relation_from_obj(arg_a, arg_b, parse_json_object(raw))
    except Exception as ex:
        log.warning('%s giving up: %s', tag, ex)
        return _relation_failure(arg_a, arg_b, ex)


async def _classify_pair_batch(topic_text, pairs, attack_mode, topic_sem):
    """Batched path: N pairs, one LLM call. On parse failure or size mismatch,
    fall back to per-pair calls for just this batch."""
    if not pairs:
        return []
    async with topic_sem:
        tag = f'pair_batch[{len(pairs)}]'
        # Reserve ~150 tokens of response per pair. Keep a floor for tiny batches.
        budget = max(400, 180 * len(pairs))
        try:
            raw = await POOL.call(
                build_batch_relation_prompt(topic_text, pairs, attack_mode=attack_mode),
                task_tag=tag,
                max_tokens=budget,
            )
            arr = parse_json_array(raw)
        except Exception as ex:
            log.warning('%s batch failed (%s); falling back to per-pair', tag, ex)
            return await asyncio.gather(*[
                _classify_single_pair(topic_text, a, b, attack_mode) for a, b in pairs
            ])

    # Match up returned objects to pairs by the 'pair' index if present,
    # otherwise fall back to positional matching. Any missing index falls
    # back to a single-item call for just that pair.
    by_index = {}
    for obj in arr:
        if isinstance(obj, dict) and 'pair' in obj:
            try:
                by_index[int(obj['pair'])] = obj
            except (TypeError, ValueError):
                pass

    # If the model gave us positional output (no 'pair' key) and the count
    # matches, use positional.
    if not by_index and len(arr) == len(pairs):
        by_index = {i: obj for i, obj in enumerate(arr) if isinstance(obj, dict)}

    results = []
    missing = []
    for i, (a, b) in enumerate(pairs):
        obj = by_index.get(i)
        if obj is None:
            missing.append((i, a, b))
        else:
            results.append((i, _relation_from_obj(a, b, obj)))

    if missing:
        log.info('%s recovering %d missing items via single-item calls', tag, len(missing))
        filled = await asyncio.gather(*[
            _classify_single_pair(topic_text, a, b, attack_mode) for _, a, b in missing
        ])
        for (i, _, _), rec in zip(missing, filled):
            results.append((i, rec))

    results.sort(key=lambda x: x[0])
    return [r for _, r in results]


# ---------- Strength batching ------------------------------------------------

async def _score_single_arg(topic_text, arg):
    tag = f"strength[{arg['arg_id']}]"
    try:
        raw = await POOL.call(build_strength_prompt(topic_text, arg), task_tag=tag, max_tokens=200)
        data = parse_json_object(raw)
        return arg['arg_id'], {
            'strength': round(_coerce_confidence(data.get('strength', 0.5)), 3),
            'rationale': data.get('rationale', 'No rationale returned.'),
        }
    except Exception as ex:
        log.warning('%s giving up: %s', tag, ex)
        return arg['arg_id'], {
            'strength': 0.5,
            'rationale': f'SCORING_FAILED: {ex}',
            'failed': True,
        }


async def _score_arg_batch(topic_text, args, topic_sem):
    if not args:
        return []
    async with topic_sem:
        tag = f'strength_batch[{len(args)}]'
        budget = max(300, 80 * len(args))
        try:
            raw = await POOL.call(
                build_batch_strength_prompt(topic_text, args),
                task_tag=tag,
                max_tokens=budget,
            )
            arr = parse_json_array(raw)
        except Exception as ex:
            log.warning('%s batch failed (%s); falling back to per-arg', tag, ex)
            return await asyncio.gather(*[_score_single_arg(topic_text, a) for a in args])

    by_index = {}
    for obj in arr:
        if isinstance(obj, dict) and 'arg' in obj:
            try:
                by_index[int(obj['arg'])] = obj
            except (TypeError, ValueError):
                pass
    if not by_index and len(arr) == len(args):
        by_index = {i: obj for i, obj in enumerate(arr) if isinstance(obj, dict)}

    results = []
    missing = []
    for i, a in enumerate(args):
        obj = by_index.get(i)
        if obj is None:
            missing.append(a)
        else:
            results.append((a['arg_id'], {
                'strength': round(_coerce_confidence(obj.get('strength', 0.5)), 3),
                'rationale': obj.get('rationale', 'No rationale returned.'),
            }))

    if missing:
        log.info('%s recovering %d missing args via single-item calls', tag, len(missing))
        filled = await asyncio.gather(*[_score_single_arg(topic_text, a) for a in missing])
        results.extend(filled)

    return results


def _chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


## 5. Per-topic orchestration

In [ ]:
def load_stage1(path=IN_FILE):
    if not path.exists():
        raise FileNotFoundError(f'Stage 1 input not found: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


async def process_topic_async(topic_rec, attack_mode='targeted', confidence_threshold=0.65):
    """Run prefilter + batched strength scoring + batched pair classification."""
    args = topic_rec['arguments']
    topic_text = topic_rec['topic_text']
    topic_sem = asyncio.Semaphore(TOPIC_CONCURRENCY)

    t0 = time.monotonic()

    # Prefilter pairs via embedding similarity.
    sim = compute_similarity_matrix(args) if PREFILTER_ENABLED else None
    keep_pairs, auto_none = prefilter_pairs(args, sim)

    total_ordered = len(args) * max(len(args) - 1, 0)
    log.info(
        'topic %s: %d args, %d ordered pairs, prefilter keeps %d, auto-None %d',
        topic_rec['topic_id'], len(args), total_ordered, len(keep_pairs), len(auto_none),
    )

    # Batched strength scoring (all arguments)
    strength_batches = list(_chunks(args, STRENGTH_BATCH_SIZE))
    strength_coros = [_score_arg_batch(topic_text, batch, topic_sem) for batch in strength_batches]

    # Batched pair classification (only kept pairs)
    pair_batches = list(_chunks(keep_pairs, PAIR_BATCH_SIZE))
    pair_coros = [_classify_pair_batch(topic_text, batch, attack_mode, topic_sem)
                  for batch in pair_batches]

    strength_batch_results, relation_batch_results = await asyncio.gather(
        asyncio.gather(*strength_coros) if strength_coros else asyncio.sleep(0, result=[]),
        asyncio.gather(*pair_coros) if pair_coros else asyncio.sleep(0, result=[]),
    )

    # Flatten
    strength_pairs = []
    for batch_result in strength_batch_results:
        strength_pairs.extend(batch_result)
    strength_scores = dict(strength_pairs)

    relations = list(auto_none)  # prefiltered None's come first, no call made
    for batch_result in relation_batch_results:
        relations.extend(batch_result)

    elapsed = time.monotonic() - t0

    # Summary
    counts = {k: 0 for k in LABELS}
    kept = 0
    filtered = 0
    failed_pairs = 0
    prefiltered_pairs = 0
    for r in relations:
        counts[r.get('label', 'None')] += 1
        kept += int(r.get('kept', False))
        filtered += int(not r.get('kept', False))
        failed_pairs += int(r.get('failed', False))
        prefiltered_pairs += int(r.get('prefiltered', False))
    failed_strengths = sum(1 for s in strength_scores.values() if s.get('failed'))
    avg_strength = (
        round(statistics.mean(v['strength'] for v in strength_scores.values()), 3)
        if strength_scores else 0.0
    )

    log.info(
        'topic %s done in %.1fs | pairs total=%d llm_classified=%d prefiltered=%d '
        'kept=%d failed_pairs=%d failed_strengths=%d',
        topic_rec['topic_id'], elapsed, len(relations), len(keep_pairs),
        prefiltered_pairs, kept, failed_pairs, failed_strengths,
    )

    return {
        'topic_id': topic_rec['topic_id'],
        'topic_text': topic_rec['topic_text'],
        'domain': topic_rec.get('domain'),
        'benchmark_label': topic_rec.get('benchmark_label'),
        'source_dataset': topic_rec.get('source_dataset'),
        'source_ref': topic_rec.get('source_ref'),
        'evaluation_split': topic_rec.get('evaluation_split', EVAL_SPLIT),
        'run_name': topic_rec.get('run_name'),
        'arguments': args,
        'argument_strength': strength_scores,
        'relations': relations,
        'summary': {
            'n_arguments': len(args),
            'n_ordered_pairs': total_ordered,
            'n_llm_classified': len(keep_pairs),
            'n_prefiltered': prefiltered_pairs,
            'kept_relations': kept,
            'filtered_relations': filtered,
            'failed_pairs': failed_pairs,
            'failed_strengths': failed_strengths,
            'label_counts': counts,
            'attack_mode': attack_mode,
            'confidence_threshold': confidence_threshold,
            'avg_strength': avg_strength,
            'elapsed_seconds': round(elapsed, 2),
        },
    }


# MLflow — optional, unchanged.
try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('MAJ-Debate')
    MLFLOW_OK = True
except Exception as ex:
    MLFLOW_OK = False
    print(f'MLflow unavailable ({ex}) - results saved locally only')


def mlflow_log(run_name, params, metrics, artifact_paths):
    if not MLFLOW_OK:
        return
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for path in artifact_paths:
            mlflow.log_artifact(str(path), artifact_path=f'stage2/{EVAL_SPLIT}')


## 6. Run Stage 2 (with resume + atomic checkpoints)

In [ ]:
stage1_doc = load_stage1()
topics = stage1_doc['topics'][:TOPIC_LIMIT] if TOPIC_LIMIT > 0 else stage1_doc['topics']
run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f'stage2-{EVAL_SPLIT}-{run_ts}'


def compute_stage2_summary(topic_payloads):
    if not topic_payloads:
        return {'total_topics': 0, 'total_arguments': 0, 'total_pairs': 0,
                'kept_relations': 0, 'filtered_relations': 0, 'avg_strength': 0.0}
    return {
        'total_topics': len(topic_payloads),
        'total_arguments': sum(t['summary']['n_arguments'] for t in topic_payloads),
        'total_pairs': sum(t['summary']['n_ordered_pairs'] for t in topic_payloads),
        'total_llm_classified': sum(t['summary'].get('n_llm_classified', 0) for t in topic_payloads),
        'total_prefiltered': sum(t['summary'].get('n_prefiltered', 0) for t in topic_payloads),
        'kept_relations': sum(t['summary']['kept_relations'] for t in topic_payloads),
        'filtered_relations': sum(t['summary']['filtered_relations'] for t in topic_payloads),
        'failed_pairs': sum(t['summary'].get('failed_pairs', 0) for t in topic_payloads),
        'failed_strengths': sum(t['summary'].get('failed_strengths', 0) for t in topic_payloads),
        'avg_strength': round(
            statistics.mean(t['summary']['avg_strength'] for t in topic_payloads), 3
        ),
    }


def save_stage2_state(topic_results_by_id, failures):
    ordered_topics = [topic_results_by_id[t['topic_id']] for t in topics if t['topic_id'] in topic_results_by_id]
    output_doc = {
        'stage': 2,
        'run_name': run_name,
        'timestamp': run_ts,
        'config': {
            'endpoints': [{'name': e['name'], 'type': e['type'], 'model': e['model']}
                          for e in ENDPOINT_CONFIGS],
            'evaluation_split': EVAL_SPLIT,
            'attack_mode': ATTACK_MODE,
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'source_stage1_run': stage1_doc.get('run_name'),
            'resume_existing': RESUME_EXISTING,
            'topic_concurrency': TOPIC_CONCURRENCY,
            'task_max_attempts': MAX_TASK_ATTEMPTS,
            'prefilter_enabled': PREFILTER_ENABLED,
            'prefilter_max_round_gap': PF_MAX_ROUND_GAP,
            'prefilter_min_sim': PF_MIN_SIMILARITY,
            'pair_batch_size': PAIR_BATCH_SIZE,
            'strength_batch_size': STRENGTH_BATCH_SIZE,
        },
        'topics': ordered_topics,
        'summary': compute_stage2_summary(ordered_topics),
    }
    tmp_out = OUT_FILE.with_suffix('.json.tmp')
    with open(tmp_out, 'w', encoding='utf-8') as f:
        json.dump(output_doc, f, indent=2)
    tmp_out.replace(OUT_FILE)

    tmp_fail = FAILURES_FILE.with_suffix('.json.tmp')
    with open(tmp_fail, 'w', encoding='utf-8') as f:
        json.dump({'run_name': run_name, 'timestamp': run_ts, 'failed_topics': failures}, f, indent=2)
    tmp_fail.replace(FAILURES_FILE)
    return output_doc


# Resume setup
existing_topics_by_id = {}
if RESUME_EXISTING and OUT_FILE.exists():
    try:
        with open(OUT_FILE, 'r', encoding='utf-8') as f:
            existing_doc = json.load(f)
        for topic_payload in existing_doc.get('topics', []):
            tid = topic_payload.get('topic_id')
            if tid and topic_payload.get('relations'):
                existing_topics_by_id[tid] = topic_payload
        print(f'Resuming from existing output: {len(existing_topics_by_id)} completed topics found')
    except Exception as ex:
        print(f'Could not resume from existing output ({ex}); starting fresh')
        existing_topics_by_id = {}

existing_failures_by_id = {}
if FAILURES_FILE.exists():
    try:
        with open(FAILURES_FILE, 'r', encoding='utf-8') as f:
            existing_failures_doc = json.load(f)
        for fp in existing_failures_doc.get('failed_topics', []):
            ftid = fp.get('topic_id')
            if ftid and ftid not in existing_topics_by_id:
                existing_failures_by_id[ftid] = fp
        if existing_failures_by_id:
            print(f'Retrying unresolved failures from prior runs: {len(existing_failures_by_id)} topics')
    except Exception as ex:
        print(f'Could not load existing failures ({ex}); continuing fresh')
        existing_failures_by_id = {}


async def run_stage2():
    topic_results_by_id = dict(existing_topics_by_id)
    failures_by_id = dict(existing_failures_by_id)

    run_start = time.monotonic()
    try:
        for idx, topic in enumerate(topics, start=1):
            topic_id = topic['topic_id']
            if topic_id in topic_results_by_id:
                print(f"[{idx}/{len(topics)}] {topic_id}: already completed, skipping")
                continue
            print(f"[{idx}/{len(topics)}] {topic_id}: {topic['topic_text']}")
            try:
                result = await process_topic_async(
                    topic,
                    attack_mode=ATTACK_MODE,
                    confidence_threshold=CONFIDENCE_THRESHOLD,
                )
                topic_results_by_id[topic_id] = result
                failures_by_id.pop(topic_id, None)
                save_stage2_state(topic_results_by_id, list(failures_by_id.values()))
                s = result['summary']
                print(
                    f"  checkpoint saved | {s['elapsed_seconds']}s | "
                    f"kept={s['kept_relations']}/{s['n_ordered_pairs']} "
                    f"(llm_calls_made={s.get('n_llm_classified', '?')}, "
                    f"prefiltered={s.get('n_prefiltered', '?')})"
                )
            except Exception as ex:
                log.exception('Stage 2 failed for %s', topic_id)
                failures_by_id[topic_id] = {
                    'topic_id': topic_id,
                    'topic_text': topic['topic_text'],
                    'error': str(ex),
                    'run_name': run_name,
                }
                save_stage2_state(topic_results_by_id, list(failures_by_id.values()))
                if not CONTINUE_ON_ERROR:
                    raise
    finally:
        await POOL.aclose()

    out = save_stage2_state(topic_results_by_id, list(failures_by_id.values()))
    total_elapsed = time.monotonic() - run_start
    print(f'\nRun finished in {total_elapsed:.1f}s')
    print('Endpoint stats:')
    for s in POOL.stats():
        status = 'DISABLED' if s['disabled'] else ('quarantined' if s['quarantined'] else 'ok')
        tail = f" reason={s['disabled_reason']}" if s['disabled'] else ''
        print(f"  {s['name']:20s} ok={s['ok']:5d} fail={s['fail']:4d} "
              f"status={status:11s} level={s['quarantine_level']}{tail}")
    return out, failures_by_id


# Bulletproof async entry (Jupyter + plain Python + Colab)
def _run_coro_bulletproof(coro):
    import threading
    try:
        asyncio.get_running_loop()
        loop_running = True
    except RuntimeError:
        loop_running = False

    if not loop_running:
        return asyncio.run(coro)

    try:
        import nest_asyncio
    except ImportError:
        try:
            import subprocess, sys
            print('Installing nest_asyncio for notebook execution...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nest_asyncio'])
            import nest_asyncio
        except Exception as ex:
            print(f'nest_asyncio unavailable ({ex}); using background-thread runner')
            nest_asyncio = None

    if nest_asyncio is not None:
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(coro)

    result_box: dict = {}

    def _thread_target():
        try:
            new_loop = asyncio.new_event_loop()
            asyncio.set_event_loop(new_loop)
            try:
                result_box['value'] = new_loop.run_until_complete(coro)
            finally:
                new_loop.close()
        except BaseException as ex:
            result_box['error'] = ex

    t = threading.Thread(target=_thread_target, daemon=False)
    t.start()
    t.join()
    if 'error' in result_box:
        raise result_box['error']
    return result_box['value']


output_doc, failures_by_id = _run_coro_bulletproof(run_stage2())

mlflow_log(
    run_name=run_name,
    params={k: (json.dumps(v) if isinstance(v, (list, dict)) else v)
            for k, v in output_doc['config'].items()},
    metrics={k: float(v) for k, v in output_doc['summary'].items() if isinstance(v, (int, float))},
    artifact_paths=[OUT_FILE, FAILURES_FILE],
)
print(f'Saved: {OUT_FILE.resolve()}')
print(output_doc['summary'])
print(f'Failed topics: {len(failures_by_id)}')


## 7. Sample output

In [ ]:
sample = output_doc['topics'][0]
print('Topic   :', sample['topic_text'])
print('Label   :', sample['benchmark_label'])
print('Args    :', sample['summary']['n_arguments'])
print('Pairs   :', sample['summary']['n_ordered_pairs'])
print('LLM     :', sample['summary'].get('n_llm_classified', 'n/a'))
print('Prefilt :', sample['summary'].get('n_prefiltered', 'n/a'))
print()
# First non-prefiltered relation, if any
for r in sample['relations']:
    if not r.get('prefiltered'):
        print(json.dumps(r, indent=2))
        break
